# No Nonsense Linear Regression: Predicting Response Times from Consumer Complaint Text

Every day, thousands of consumers file complaints with the CFPB about financial products. Each complaint contains a narrative — raw text describing what went wrong. Some complaints get a response in a day. Others take weeks.

**The question:** Can the *text* of a complaint tell us anything about how long it takes a company to respond?

We'll extract NLP features from complaint narratives, throw them into a linear regression, and see what sticks. Along the way, we'll cover the stuff most tutorials skip: assumption violations, coefficient interpretation with text features, and what to do when your R² is underwhelming (spoiler: that's normal with real data).

**What you'll need:**
- The CFPB complaint sample (see `data/README.md` for download instructions)
- Python 3.10+ with the packages in `requirements.txt`
- ~15 minutes

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

%matplotlib inline

## 1. The Data

The CFPB Consumer Complaints Database is public, free, and messy in all the right ways. We've already sampled 50K complaints that have both narrative text and a valid response time. Let's load it.

In [ ]:
data_path = Path("../data/complaints_sample.csv")

if not data_path.exists():
    raise FileNotFoundError(
        "Sample data not found. Run 'python data/prepare_data.py' first. "
        "See data/README.md for download instructions."
    )

df = pd.read_csv(data_path, parse_dates=["Date received", "Date sent to company"])
print(f"Shape: {df.shape}")
df.head(3)

In [ ]:
# Our target: how many days between CFPB receiving the complaint and forwarding it to the company
df["response_time_days"].describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw distribution
axes[0].hist(df["response_time_days"], bins=50, edgecolor="black", alpha=0.7)
axes[0].set_xlabel("Response Time (days)")
axes[0].set_ylabel("Count")
axes[0].set_title("Raw Distribution")

# Zoomed in (drop extreme outliers for visualization)
reasonable = df[df["response_time_days"] <= 30]
axes[1].hist(reasonable["response_time_days"], bins=30, edgecolor="black", alpha=0.7)
axes[1].set_xlabel("Response Time (days)")
axes[1].set_ylabel("Count")
axes[1].set_title(f"Zoomed: ≤30 days ({len(reasonable)/len(df)*100:.1f}% of data)")

plt.tight_layout()
plt.show()

That spike at 0-1 days? Most complaints get forwarded immediately. The long tail is where it gets interesting.

For modeling, we'll cap response time at 30 days. Anything beyond that is likely a process anomaly, not something the complaint *text* can explain.

In [ ]:
# Filter to reasonable response times
df = df[df["response_time_days"] <= 30].copy()
print(f"After filtering: {len(df):,} records")
print(f"Mean response time: {df['response_time_days'].mean():.2f} days")
print(f"Median response time: {df['response_time_days'].median():.1f} days")

In [ ]:
# Quick look at what we're working with
sample_text = df["Consumer complaint narrative"].iloc[0]
print(f"Sample complaint ({len(sample_text)} chars):")
print(sample_text[:500] + "..." if len(sample_text) > 500 else sample_text)

## 2. Feature Engineering from Text

Here's where the NLP meets the regression. We need to turn raw complaint text into numbers that a linear model can work with.

We'll extract four types of features:
1. **Basic text stats** — length, word count, sentence count
2. **Readability** — how complex is the writing?
3. **Sentiment** — how negative/emotional is the complaint?
4. **TF-IDF** — which specific words matter?

Each of these is a hypothesis: *does this text property predict how long a company takes to respond?*

In [ ]:
# Basic text features — looks simple but these are surprisingly informative
df["text_length"] = df["Consumer complaint narrative"].str.len()
df["word_count"] = df["Consumer complaint narrative"].str.split().str.len()
df["sentence_count"] = df["Consumer complaint narrative"].str.count(r'[.!?]+')
df["avg_word_length"] = df["text_length"] / df["word_count"]
df["words_per_sentence"] = df["word_count"] / df["sentence_count"].replace(0, 1)

print("Basic text features:")
df[["text_length", "word_count", "sentence_count", "avg_word_length", "words_per_sentence"]].describe().round(1)

In [ ]:
import textstat

# Readability scores — how complex is the writing?
# Flesch Reading Ease: higher = easier to read (60-70 is plain English)
df["flesch_reading_ease"] = df["Consumer complaint narrative"].apply(textstat.flesch_reading_ease)

# Flesch-Kincaid Grade Level: roughly maps to US school grade level
df["flesch_kincaid_grade"] = df["Consumer complaint narrative"].apply(textstat.flesch_kincaid_grade)

print("Readability scores:")
df[["flesch_reading_ease", "flesch_kincaid_grade"]].describe().round(1)

In [ ]:
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer

nltk.download("vader_lexicon", quiet=True)
sia = SentimentIntensityAnalyzer()

# VADER sentiment — designed for social media text, works well enough on complaints
sentiment_scores = df["Consumer complaint narrative"].apply(lambda x: sia.polarity_scores(x))
df["sentiment_neg"] = sentiment_scores.apply(lambda x: x["neg"])
df["sentiment_neu"] = sentiment_scores.apply(lambda x: x["neu"])
df["sentiment_pos"] = sentiment_scores.apply(lambda x: x["pos"])
df["sentiment_compound"] = sentiment_scores.apply(lambda x: x["compound"])

print("Sentiment scores:")
df[["sentiment_neg", "sentiment_neu", "sentiment_pos", "sentiment_compound"]].describe().round(3)

No surprise — complaints skew negative. But there's variance, and that's what we need.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# TF-IDF: the workhorse of text-to-numbers
# We'll limit to 50 features to keep the model interpretable
tfidf = TfidfVectorizer(
    max_features=50,
    stop_words="english",
    min_df=50,       # appear in at least 50 docs
    max_df=0.95,     # not in >95% of docs
    ngram_range=(1, 2),
)

tfidf_matrix = tfidf.fit_transform(df["Consumer complaint narrative"])
tfidf_feature_names = [f"tfidf_{name}" for name in tfidf.get_feature_names_out()]
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf_feature_names, index=df.index)

print(f"TF-IDF features: {len(tfidf_feature_names)}")
print(f"Top terms: {', '.join(tfidf.get_feature_names_out()[:10])}...")

In [ ]:
# Combine all features
nlp_features = [
    "text_length", "word_count", "sentence_count", "avg_word_length", "words_per_sentence",
    "flesch_reading_ease", "flesch_kincaid_grade",
    "sentiment_neg", "sentiment_neu", "sentiment_pos", "sentiment_compound",
]

X = pd.concat([df[nlp_features], tfidf_df], axis=1)
y = df["response_time_days"]

print(f"Feature matrix: {X.shape}")
print(f"Target: {y.shape}")

In [ ]:
# Quick look at correlations between our hand-crafted features and the target
correlations = df[nlp_features + ["response_time_days"]].corr()["response_time_days"].drop("response_time_days").sort_values()

fig, ax = plt.subplots(figsize=(10, 6))
correlations.plot(kind="barh", ax=ax, color=["steelblue" if v > 0 else "coral" for v in correlations])
ax.set_xlabel("Pearson Correlation with Response Time")
ax.set_title("Feature Correlations with Response Time")
ax.axvline(x=0, color="black", linewidth=0.5)
plt.tight_layout()
plt.show()

print("\nDon't panic if these are small. With real-world text data,")
print("correlations of 0.05-0.15 are typical. We're not predicting stock prices here.")

## 3. Building the Model

No ceremony. Split the data, fit the model, see what happens.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features — important when mixing text length (100s) with TF-IDF (0-1)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Fit
model = LinearRegression()
model.fit(X_train_scaled, y_train)

# Evaluate
y_pred = model.predict(X_test_scaled)

print(f"R² (test):  {r2_score(y_test, y_pred):.4f}")
print(f"RMSE:       {np.sqrt(mean_squared_error(y_test, y_pred)):.4f} days")
print(f"MAE:        {mean_absolute_error(y_test, y_pred):.4f} days")
print(f"\nBaseline (always predict mean): MAE = {(y_test - y_test.mean()).abs().mean():.4f} days")

**Let's talk about that R².**

If you're disappointed, good — you're paying attention. A low R² on real-world text data is *expected*. Here's why:

1. Response time is driven primarily by *company processes*, not complaint text
2. We're using surface-level NLP features — they capture writing style, not content depth
3. There's massive noise in the target (holidays, staffing, backlogs)

The value isn't in the R². It's in the *coefficients* — what text features are associated with longer or shorter response times, even if the effect is small. That's the story.

In [ ]:
# Residual plots — the real diagnostic
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

residuals = y_test - y_pred

# Predicted vs Actual
axes[0].scatter(y_pred, y_test, alpha=0.1, s=5)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--", linewidth=2)
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")
axes[0].set_title("Predicted vs Actual")

# Residuals vs Predicted
axes[1].scatter(y_pred, residuals, alpha=0.1, s=5)
axes[1].axhline(y=0, color="r", linestyle="--")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("Residuals")
axes[1].set_title("Residuals vs Predicted")

# Residual distribution
axes[2].hist(residuals, bins=50, edgecolor="black", alpha=0.7)
axes[2].set_xlabel("Residual")
axes[2].set_ylabel("Count")
axes[2].set_title("Residual Distribution")

plt.tight_layout()
plt.show()

## 4. Interpreting the Coefficients

This is where linear regression earns its keep. Unlike a black-box model, we can look at every coefficient and say *exactly* what it means.

In [ ]:
# Coefficient table
coef_df = pd.DataFrame({
    "feature": X.columns,
    "coefficient": model.coef_,
    "abs_coefficient": np.abs(model.coef_),
}).sort_values("abs_coefficient", ascending=False)

print("Top 15 features by absolute coefficient (standardized):")
print("(Coefficient = change in predicted response time per 1 SD change in feature)")
print()
coef_df.head(15)[["feature", "coefficient"]].to_string(index=False)

In [ ]:
# Visualize top coefficients
top_n = 20
top_features = coef_df.head(top_n)

fig, ax = plt.subplots(figsize=(10, 8))
colors = ["coral" if c > 0 else "steelblue" for c in top_features["coefficient"]]
ax.barh(range(top_n), top_features["coefficient"].values, color=colors)
ax.set_yticks(range(top_n))
ax.set_yticklabels(top_features["feature"].values)
ax.set_xlabel("Standardized Coefficient")
ax.set_title(f"Top {top_n} Features by Coefficient Magnitude")
ax.axvline(x=0, color="black", linewidth=0.5)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print("Coral = longer response time | Blue = shorter response time")

In [ ]:
import statsmodels.api as sm

# For statistical significance, we need statsmodels
X_train_sm = sm.add_constant(X_train_scaled)
ols_model = sm.OLS(y_train, X_train_sm).fit()

# Summary of just the significant features
results_df = pd.DataFrame({
    "feature": ["intercept"] + list(X.columns),
    "coefficient": ols_model.params,
    "std_error": ols_model.bse,
    "t_stat": ols_model.tvalues,
    "p_value": ols_model.pvalues,
})

significant = results_df[results_df["p_value"] < 0.05].sort_values("p_value")
print(f"Significant features (p < 0.05): {len(significant) - 1} out of {len(X.columns)}")
print()
significant.head(15).to_string(index=False)

**Reading these coefficients:**

Since we standardized the features, each coefficient tells you: *"A 1 standard deviation increase in this feature is associated with X days change in response time."*

With 50K observations, many features will be statistically significant even if the effect is tiny. That's the difference between *statistical* significance and *practical* significance. A coefficient of 0.01 days is significant with enough data, but it means nothing in practice.

## 5. What to Watch Out For

Here's what most tutorials skip. Linear regression makes assumptions. Let's check if ours hold — and what to do when they don't.

In [ ]:
from scipy import stats

# Assumption 1: Normality of residuals
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Q-Q plot
stats.probplot(residuals, dist="norm", plot=axes[0])
axes[0].set_title("Q-Q Plot of Residuals")

# Shapiro-Wilk on a subsample (full dataset is too large for the test)
_, sw_p = stats.shapiro(residuals.sample(min(5000, len(residuals)), random_state=42))
axes[1].hist(residuals, bins=50, density=True, edgecolor="black", alpha=0.7)
x_range = np.linspace(residuals.min(), residuals.max(), 100)
axes[1].plot(x_range, stats.norm.pdf(x_range, residuals.mean(), residuals.std()), "r-", linewidth=2)
axes[1].set_title(f"Residuals vs Normal (Shapiro-Wilk p={sw_p:.4f})")
axes[1].set_xlabel("Residual")

plt.tight_layout()
plt.show()

print("The residuals aren't normal. With count-like target data (days), they rarely are.")
print("This doesn't invalidate the model — it means confidence intervals and p-values are approximate.")
print("With 50K observations, the Central Limit Theorem has your back for coefficient estimates.")

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Assumption 2: No multicollinearity
# Check VIF for hand-crafted features (not TF-IDF — those are sparse and numerous)
vif_data = pd.DataFrame()
X_vif = df[nlp_features].dropna()
vif_data["feature"] = nlp_features
vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
vif_data = vif_data.sort_values("VIF", ascending=False)

print("Variance Inflation Factors (VIF > 10 = problematic):")
print()
print(vif_data.to_string(index=False))
print()
print("High VIF between text_length and word_count? Obviously — longer text has more words.")
print("You could drop one, but keeping both doesn't bias coefficients — it just inflates their variance.")

In [ ]:
# Assumption 3: Homoscedasticity — constant variance of residuals
fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(y_pred, np.abs(residuals), alpha=0.1, s=5)

# Add a trend line to make the pattern clear
bins = pd.cut(y_pred, bins=20)
binned = pd.DataFrame({"pred": y_pred, "abs_resid": np.abs(residuals), "bin": bins})
bin_means = binned.groupby("bin", observed=True).agg({"pred": "mean", "abs_resid": "mean"}).dropna()
ax.plot(bin_means["pred"], bin_means["abs_resid"], "r-o", linewidth=2, markersize=5)

ax.set_xlabel("Predicted Response Time")
ax.set_ylabel("|Residual|")
ax.set_title("Checking Homoscedasticity")
plt.tight_layout()
plt.show()

print("If that red line is flat, great. If it slopes up, you have heteroscedasticity.")
print("With response time data, some heteroscedasticity is expected.")
print("Fix: use robust standard errors (statsmodels HC3) or transform the target.")

In [ ]:
# Robust standard errors — the practical fix for heteroscedasticity
ols_robust = sm.OLS(y_train, X_train_sm).fit(cov_type="HC3")

# Compare: which features change significance with robust SEs?
comparison = pd.DataFrame({
    "feature": ["intercept"] + list(X.columns),
    "p_value_ols": ols_model.pvalues,
    "p_value_robust": ols_robust.pvalues,
})

# Features that lose significance with robust SEs
flipped = comparison[
    (comparison["p_value_ols"] < 0.05) & (comparison["p_value_robust"] >= 0.05)
]
if len(flipped) > 0:
    print("Features that LOSE significance with robust standard errors:")
    print(flipped[["feature", "p_value_ols", "p_value_robust"]].to_string(index=False))
    print("\nThese were false positives — their significance depended on the homoscedasticity assumption.")
else:
    print("All significant features remain significant with robust standard errors. Good.")

### Common Pitfalls with NLP + Regression

**1. Too many TF-IDF features.** We used 50. If you crank it to 5,000, you'll overfit and your coefficients become meaningless. With linear regression, interpretability *is* the feature. Don't sacrifice it.

**2. Data leakage.** If any of your text features encode information about the *response* (not the complaint), you've got leakage. For example, if complaints mention "I waited 3 weeks" — that's the target leaking into the features.

**3. Confusing correlation with causation.** A longer complaint doesn't *cause* slower response. It might *correlate* with complaint complexity, which correlates with response time. Linear regression gives you associations, not causal effects.

**4. Ignoring the intercept.** The intercept is the predicted response time when all features are at their mean (since we standardized). It's your baseline — don't ignore it.

## 6. So What?

Here's what we learned:

**About the data:**
- Complaint text features have weak but real associations with response time
- The model's R² is low because response time is driven by company operations, not complaint content — and that's fine
- Specific words (from TF-IDF) are often more predictive than aggregate features like sentiment

**About linear regression with text data:**
- It works as an interpretability tool, not a prediction powerhouse
- Feature engineering matters more than model complexity
- Always check your assumptions — they *will* be violated with real data. The question is how badly.
- Robust standard errors are your friend. Use them by default.

**When to use this approach:**
- When you need to *explain* what text features matter, not just predict
- When stakeholders need a story, not a black box
- As a baseline before throwing deep learning at a problem

**When NOT to use it:**
- When prediction accuracy is all that matters (use gradient boosting or transformers)
- When your text features are all TF-IDF with thousands of dimensions (use regularization: Ridge/Lasso)
- When the relationship between text and target is clearly non-linear

In [ ]:
# Final summary
print("=" * 60)
print("MODEL SUMMARY")
print("=" * 60)
print(f"Records:          {len(X):,}")
print(f"Features:         {X.shape[1]} ({len(nlp_features)} hand-crafted + {len(tfidf_feature_names)} TF-IDF)")
print(f"R² (test):        {r2_score(y_test, y_pred):.4f}")
print(f"RMSE:             {np.sqrt(mean_squared_error(y_test, y_pred)):.2f} days")
print(f"MAE:              {mean_absolute_error(y_test, y_pred):.2f} days")
print(f"Significant vars: {len(significant) - 1}/{len(X.columns)}")
print("=" * 60)